# Cleanup notebook: .NET controller ⇄ Angular service training pairs

Builds these training sets from a local repo (`First/`), all in one pipeline:

| # | Task | Prompt | Response |
|---|------|--------|----------|
| 1 | `backend_to_frontend` | controller file | Angular service file that calls it |
| 2 | `frontend_to_backend` | Angular service file | controller file |
| 3 | `dto_to_ts` | C# class | real TypeScript model (`Guid`→`string`, `decimal`→`number`, `List<T>`→`T[]` ...) |
| 4 | `endpoint_to_method` / `method_to_endpoint` | one action | the one service method that calls it (and back) |
| 5 | `find_the_bug` | endpoint + service method that do not match | explanation + fix (**real** findings and **mutation-verified** ones) |
| 6 | `feature_scaffold` | short spec | controller + DTOs + service + models for one feature |

Section 6 is the **consistency check** that powers task 5. It is also a validation step you can run on any repo.

Only `pandas` and the standard library are needed (git is called through `subprocess`).

## 1. Config

In [1]:
import re, json, hashlib, fnmatch, subprocess, textwrap
from pathlib import Path
from collections import defaultdict
import pandas as pd



try:
    from IPython.display import display
except ImportError:                      # running as a plain script
    display = print

ROOT    = Path("First")                  # repo folder (contains .git, or sits inside it)
OUT_DIR = Path("out"); OUT_DIR.mkdir(exist_ok=True)

SYSTEM_MSG = ("You are a coding assistant for the ACWA project. Stack: ASP.NET Core Web API (C#) and an "
              "Angular client that uses the legacy @angular/http service style. Follow the project's conventions.")

MAX_TOKENS        = 2048                 # examples above this are flagged too_long (approximate count)
TEST_FEATURES     = {"accounts"}         # whole features held out for testing (never split a feature)
INCLUDE_PATH_HINT = True                 # add "Target file: ..." to prompts (teaches folder conventions)
MUTATION_KINDS    = ("url_typo", "wrong_verb", "drop_field")

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)


## 2. Collect files (ignore rules) + git history per file

In [2]:
IGNORE_DIRS = {".git", ".github", ".vs", ".vscode", ".idea", "bin", "obj", "packages", "testresults", "publish",
               "node_modules", "dist", ".angular", "coverage", "out-tsc", ".nx", ".cache", "migrations",
               "__pycache__", ".venv", "venv"}
IGNORE_FILE_PATTERNS = [
    "*.g.cs", "*.g.i.cs", "*.designer.cs", "*.generated.cs", "*.assemblyinfo.cs", "*modelsnapshot.cs",
    "*.min.js", "*.min.css", "*.map", "*.bundle.js",
    "package-lock.json", "yarn.lock", "pnpm-lock.yaml", "*.lock",
    "appsettings*.json", ".env", ".env.*", "*.pfx", "*.key", "*.pem", "*.p12", "secrets.json",
    "launchsettings.json", "*.user", "*.csproj", "*.sln",
    "*.dll", "*.exe", "*.pdb", "*.png", "*.jpg", "*.jpeg", "*.gif", "*.svg", "*.ico", "*.woff", "*.woff2",
    "*.ttf", "*.eot", "*.pdf", "*.zip", "*.log", "*.tmp", "*.bak", ".ds_store", "thumbs.db",
    "polyfills.ts", "main.ts", "test.ts", "environment*.ts",
]
ALLOWED_EXTENSIONS = {".cs", ".ts", ".html", ".scss", ".css", ".md"}
MAX_FILE_BYTES = 200_000

def should_ignore(path: Path, root: Path) -> bool:
    rel = path.relative_to(root)
    if any(part.lower() in IGNORE_DIRS for part in rel.parts[:-1]):          return True
    if path.suffix.lower() not in ALLOWED_EXTENSIONS:                         return True
    if any(fnmatch.fnmatch(path.name.lower(), pat) for pat in IGNORE_FILE_PATTERNS): return True
    try:    return path.stat().st_size > MAX_FILE_BYTES
    except OSError: return True

def collect_files(root):
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.is_file() and not should_ignore(p, root))

def git_history(root: Path):
    """-> (repo_top_dir, {repo-relative posix path: [(short_hash, author, iso_date, subject), ...] newest first})"""
    try:
        top = subprocess.run(["git", "-C", str(root), "rev-parse", "--show-toplevel"],
                             capture_output=True, text=True, check=True).stdout.strip()
        raw = subprocess.run(["git", "-C", str(root), "-c", "core.quotepath=off", "log", "--name-only",
                              "--pretty=format:@@%H|%an|%aI|%s"],
                             capture_output=True, text=True, encoding="utf-8", check=True).stdout
    except subprocess.CalledProcessError as e:
        print("git failed -> history columns will be empty:", (e.stderr or "").strip()[:200])
        return Path(root).resolve(), {}
    except FileNotFoundError:
        print("git is not installed / not on PATH -> history columns will be empty.")
        return Path(root).resolve(), {}
    hist, cur = defaultdict(list), None
    for line in raw.splitlines():
        if line.startswith("@@"):
            h, a, d, s = line[2:].split("|", 3); cur = (h[:7], a, d, s)
        elif line.strip() and cur:
            hist[line.strip()].append(cur)
    return Path(top).resolve(), hist

def classify(rel: str) -> str:
    n = rel.lower()
    if n.endswith(".spec.ts"):                          return "ts_spec"
    if n.endswith(".cs") and "/controllers/" in n:      return "cs_controller"
    if n.endswith(".cs"):                               return "cs_other"
    if n.endswith(".service.ts"):                       return "ts_service"
    if n.endswith(".model.ts"):                         return "ts_model"
    if n.endswith(".component.ts"):                     return "ts_component"
    return "other"

files = collect_files(ROOT)
repo_top, history = git_history(ROOT)
commits = [history.get(f.resolve().relative_to(repo_top).as_posix(), []) for f in files]

df_files = pd.DataFrame({
    "file_path":  [f.relative_to(ROOT).as_posix() for f in files],
    "extension":  [f.suffix.lower() for f in files],
    "kind":       [classify(f.relative_to(ROOT).as_posix()) for f in files],
    "size_bytes": [f.stat().st_size for f in files],
    "n_commits":  [len(c) for c in commits],
    "n_authors":  [len({x[1] for x in c}) for c in commits],
    "last_commit_date": [c[0][2] if c else None for c in commits],
    "last_commit_msg":  [c[0][3] if c else None for c in commits],
})
print(f"{len(df_files)} files kept")
display(df_files["kind"].value_counts().to_frame("files"))
display(df_files.head(20))

102 files kept


,files
kind,
cs_other,43
other,28
ts_spec,12
ts_component,9
ts_model,5
cs_controller,3
ts_service,2


,file_path,extension,kind,size_bytes,n_commits,n_authors,last_commit_date,last_commit_msg
0,ACWA.Domain/ACWAContext.cs,.cs,cs_other,1236,1,1,2019-01-15T20:53:30+02:00,Add project files.
1,ACWA.Domain/EF/AuctionContext.cs,.cs,cs_other,1405,1,1,2019-01-15T20:53:30+02:00,Add project files.
2,ACWA.Domain/Interfaces/IRepository.cs,.cs,cs_other,372,1,1,2019-01-15T20:53:30+02:00,Add project files.
3,ACWA.Domain/Interfaces/IUnitOfWork.cs,.cs,cs_other,311,1,1,2019-01-15T20:53:30+02:00,Add project files.
4,ACWA.Domain/Models/Category.cs,.cs,cs_other,1115,1,1,2019-01-15T20:53:30+02:00,Add project files.
5,ACWA.Domain/Models/Product.cs,.cs,cs_other,845,1,1,2019-01-15T20:53:30+02:00,Add project files.
6,ACWA.Domain/Models/Profile.cs,.cs,cs_other,167,1,1,2019-01-15T20:53:30+02:00,Add project files.
7,ACWA.Domain/Models/User.cs,.cs,cs_other,457,1,1,2019-01-15T20:53:30+02:00,Add project files.
8,ACWA.Domain/Repositories/CategoryRepository.cs,.cs,cs_other,1279,1,1,2019-01-15T20:53:30+02:00,Add project files.
9,ACWA.Domain/Repositories/EFUnitOfWork.cs,.cs,cs_other,1806,1,1,2019-01-15T20:53:30+02:00,Add project files.


## 3. Parsing helpers (brace matching, attributes, comments)

In [3]:
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8-sig", errors="ignore").replace("\r\n", "\n")   # drop BOM, normalise EOL

def approx_tokens(s: str) -> int:
    return int(len(s) / 3.5) + 1             # rough; swap in a real tokenizer later

def match_brace(text, open_idx, open_ch="{", close_ch="}"):
    """Index of the bracket that closes text[open_idx]; skips strings and comments."""
    depth, i, n = 0, open_idx, len(text)
    while i < n:
        c = text[i]
        if text.startswith("//", i):
            j = text.find("\n", i); i = n if j == -1 else j; continue
        if text.startswith("/*", i):
            j = text.find("*/", i + 2); i = n if j == -1 else j + 2; continue
        if c in "\"'`":
            q = c; i += 1
            while i < n and text[i] != q:
                if text[i] == "\\": i += 1
                i += 1
            i += 1; continue
        if c == open_ch: depth += 1
        elif c == close_ch:
            depth -= 1
            if depth == 0: return i
        i += 1
    return n - 1

def split_top_level(s, sep=","):
    """Split on sep, ignoring separators inside quotes or nested <>, (), [], {}."""
    parts, cur, depth, quote, prev = [], [], 0, None, ""
    for ch in s:
        if quote:
            cur.append(ch)
            if ch == quote: quote = None
        else:
            if ch in "\"'`": quote = ch
            elif ch in "<([{": depth += 1
            elif ch in ">)]}" and not (ch == ">" and prev == "="): depth -= 1     # "=>" is not a bracket
            if ch == sep and depth == 0:
                parts.append("".join(cur).strip()); cur = []; prev = ch; continue
            cur.append(ch)
        prev = ch
    tail = "".join(cur).strip()
    if tail: parts.append(tail)
    return parts

def leading_block(text, line_start):
    """Walk upwards over directly attached [Attributes], @Decorators and // comments.
    Returns (start_index, [attribute/decorator lines])."""
    start, attrs = line_start, []
    while start > 0:
        prev_start = text.rfind("\n", 0, start - 1) + 1
        line = text[prev_start:start - 1].strip()
        if line.startswith("[") or line.startswith("@"): attrs.append(line)
        elif not line.startswith("//"): break
        start = prev_start
    return start, attrs[::-1]

def line_no(text, idx): return text.count("\n", 0, idx) + 1
def dedent(code_str):   return textwrap.dedent(code_str).strip("\n")

## 4. Parse C# (controllers → endpoints, DTO classes) and TypeScript (services → HTTP calls, models)

In [4]:
# ---------- C# ----------
CS_CLASS_RE  = re.compile(r"(?m)^[ \t]*(?:public\s+|internal\s+)?(?:static\s+|sealed\s+|abstract\s+|partial\s+)*class\s+"
                          r"(?P<name>\w+)(?P<tparams><[^>]+>)?[ \t]*(?::[ \t]*(?P<bases>[^\n{]+))?\s*\{")
CS_METHOD_RE = re.compile(r"(?m)^[ \t]*public\s+(?:virtual\s+|override\s+|static\s+)*(?:async\s+)?"
                          r"(?P<ret>[\w<>\[\],.?\s]+?)\s+(?P<name>\w+)\s*\((?P<params>[^)]*)\)\s*\{")
CS_PROP_RE   = re.compile(r"public\s+(?P<type>[\w<>\[\],.?\s]+?)\s+(?P<name>\w+)\s*\{\s*get;")
ATTR_RE      = re.compile(r'\[\s*(\w+)\s*(?:\(((?:"[^"]*"|[^\]"])*)\))?\s*\]')      # args may contain "..[controller].." strings

def parse_attrs(lines):
    out = []
    for ln in lines:
        for m in ATTR_RE.finditer(ln):
            q = re.search(r'"([^"]*)"', m.group(2) or "")
            out.append({"name": m.group(1), "args": (m.group(2) or "").strip(), "template": q.group(1) if q else None})
    return out

def parse_cs_params(params):
    out = []
    for p in split_top_level(params):
        attrs = re.findall(r"\[\s*(\w+)", p)
        clean = re.sub(r"\[[^\]]*\]\s*", "", p).strip()
        default = None
        if "=" in clean: clean, default = [x.strip() for x in clean.split("=", 1)]
        typ, _, name = clean.rpartition(" ")
        out.append({"name": name.strip(), "type": typ.strip(), "default": default,
                    "source": next((a for a in attrs if a.startswith("From")), None)})
    return out

def unwrap_return(ret):
    """Task<ActionResult<X>> -> 'X'; void/Task -> 'void'; IActionResult -> None (unknown)."""
    r = re.sub(r"\s+", "", ret)
    while True:
        m = re.fullmatch(r"(?:Task|ValueTask|ActionResult)<(.+)>", r)
        if not m: break
        r = m.group(1)
    if r in ("void", "Task", "ValueTask"): return "void"
    if r in ("IActionResult", "ActionResult", "IResult"): return None
    return r

def join_route(base, sub, controller, action):
    ctrl = controller[:-len("Controller")] if controller.endswith("Controller") else controller
    base = (base or "").replace("[controller]", ctrl.lower())
    sub  = (sub or "").replace("[action]", action.lower()).replace("[controller]", ctrl.lower())
    full = sub.lstrip("~") if sub.startswith(("/", "~/")) else "/".join(x.strip("/") for x in (base, sub) if x)
    return "/" + full.strip("/")

def norm_route(r): return re.sub(r"\{[^}]*\}", "{}", r).lower()

def parse_cs_file(text, path):
    classes, endpoints = [], []
    for cm in CS_CLASS_RE.finditer(text):
        name, open_idx = cm.group("name"), cm.end() - 1
        end = match_brace(text, open_idx)
        cstart, cattr_lines = leading_block(text, cm.start())
        cattrs = parse_attrs(cattr_lines)
        bases = (cm.group("bases") or "").strip()
        props = [{"name": p.group("name"), "type": re.sub(r"\s+", " ", p.group("type")).strip()}
                 for p in CS_PROP_RE.finditer(text, open_idx + 1, end)]
        is_controller = name.endswith("Controller") or "ControllerBase" in bases
        ctor = re.search(rf"public\s+{name}\s*\([^)]*\)", text[open_idx:end])
        info = {"name": name, "tparams": cm.group("tparams") or "", "path": path, "is_controller": is_controller,
                "props": props, "code": text[cstart:end + 1],
                "decl": "\n".join(cattr_lines + [text[cm.start():open_idx].strip()]),
                "ctor": re.sub(r"\s+", " ", ctor.group(0)) if ctor else ""}
        classes.append(info)
        if not is_controller: continue

        base_route = next((a["template"] for a in cattrs if a["name"] == "Route"), "")
        cls_auth = next((("AllowAnonymous" if a["name"] == "AllowAnonymous" else f"Authorize({a['args']})")
                         for a in cattrs if a["name"] in ("Authorize", "AllowAnonymous")), None)
        for mm in CS_METHOD_RE.finditer(text, open_idx + 1, end):
            mstart, mattr_lines = leading_block(text, mm.start())
            attrs = parse_attrs(mattr_lines)
            verb_attr = next((a for a in attrs if re.fullmatch(r"Http(Get|Post|Put|Delete|Patch)", a["name"])), None)
            if not verb_attr: continue                                     # not an endpoint
            route_attr = next((a for a in attrs if a["name"] == "Route"), None)
            sub = verb_attr["template"] if verb_attr["template"] is not None else (route_attr["template"] if route_attr else "")
            action = mm.group("name")
            full = join_route(base_route, sub, name, action)
            params = parse_cs_params(mm.group("params"))
            mend = match_brace(text, mm.end() - 1)
            route_params = re.findall(r"\{\*?(\w+)", full)
            body = next((p for p in params if p["source"] == "FromBody"), None)
            act_auth = next((("AllowAnonymous" if a["name"] == "AllowAnonymous" else f"Authorize({a['args']})")
                             for a in attrs if a["name"] in ("Authorize", "AllowAnonymous")), None)
            endpoints.append({
                "id": f"{name}.{action}[{verb_attr['name'][4:].upper()} {full}]",
                "controller": name, "controller_path": path, "action": action,
                "feature": (name[:-len("Controller")] if name.endswith("Controller") else name).lower(),
                "verb": verb_attr["name"][4:].upper(), "route": full, "route_norm": norm_route(full),
                "route_params": route_params,
                "params": params,
                "body_type": body["type"] if body else None,
                "query_params": [p["name"] for p in params if p["source"] == "FromQuery"],
                "response_type": unwrap_return(mm.group("ret")),
                "auth": act_auth or cls_auth,
                "code": text[mstart:mend + 1],
                "start_line": line_no(text, mstart), "end_line": line_no(text, mend),
                "class_decl": info["decl"], "ctor": info["ctor"], "base_route": base_route})
    return classes, endpoints

# ---------- TypeScript ----------
TS_CLASS_RE  = re.compile(r"(?m)^export\s+(?:abstract\s+)?(?P<kind>class|interface)\s+(?P<name>\w+)(?P<tparams><[^>{]+>)?[^{\n]*\{")
TS_FIELD_RE  = re.compile(r"(?m)^[ \t]*(?:public\s+|readonly\s+)*(?P<name>\w+)(?P<opt>\?)?\s*:\s*(?P<type>[^;=\n]+?)\s*(?:=[^;\n]*)?;")
TS_METHOD_RE = re.compile(r"(?m)^[ \t]*(?:public|private|protected)\s+(?:async\s+)?(?P<name>\w+)\s*\((?P<params>[^)]*)\)"
                          r"\s*(?::\s*(?P<ret>[^{\n]+?))?\s*\{")
HTTP_CALL_RE = re.compile(r"this\.http\.(?P<verb>get|post|put|delete|patch)\s*(?:<[^>(]*>)?\s*\(")

def normalize_url_expr(expr):
    """'/api/users/' + id -> ('/api/users/{}', False). Handles 'a' + b, `a/${b}` and a leading base-url variable."""
    out, dynamic_tail = "", False
    for i, p in enumerate(split_top_level(expr, "+")):
        if p[:1] in "'\"`":
            lit = p[1:-1]
            if p[0] == "`": lit = re.sub(r"\$\{[^}]*\}", "{}", lit)
            out += lit
        elif i == 0:      continue                         # base-url variable, e.g. this.apiUrl + '/users'
        elif out.endswith("/"): out += "{}"                # path parameter
        else:             dynamic_tail = True              # '/api/users' + parameters  (query string built elsewhere)
    out = re.sub(r"^\{\}(?=/)", "", out).split("?")[0]
    return "/" + out.strip("/"), dynamic_tail

def resolve_body(method_text, params, body_expr):
    """-> (declared type name, [keys of the object literal if the body is a local literal])"""
    ident = (body_expr or "").strip()
    if not re.fullmatch(r"\w+", ident): return "inline", []
    m = re.search(rf"\b{ident}\s*\??:\s*([\w<>\[\]]+)", params)
    if m: return m.group(1), []
    m = re.search(rf"(?:const|let|var)\s+{ident}\s*(?::\s*([\w<>\[\]]+))?\s*=\s*\{{", method_text)
    if m:
        lit_open = m.end() - 1
        inner = method_text[lit_open + 1:match_brace(method_text, lit_open)]
        return (m.group(1) or "object"), [p.split(":")[0].strip() for p in split_top_level(inner) if ":" in p]
    return "unknown", []

def parse_ts_file(text, path):
    models, service, calls = [], None, []
    for cm in TS_CLASS_RE.finditer(text):
        open_idx = cm.end() - 1
        end = match_brace(text, open_idx)
        cstart, decorators = leading_block(text, cm.start())
        is_service = "this.http." in text[open_idx:end]
        if is_service:
            ctor = re.search(r"constructor\s*\([^)]*\)", text[open_idx:end])
            service = {"name": cm.group("name"), "path": path,
                       "ctor": re.sub(r"\s+", " ", ctor.group(0)) if ctor else ""}
        elif not decorators:                                   # decorated classes are components/modules/guards, not models
            fields = [{"name": f.group("name"), "type": f.group("type").strip(), "optional": bool(f.group("opt"))}
                      for f in TS_FIELD_RE.finditer(text, open_idx + 1, end)]
            if fields: models.append({"name": cm.group("name"), "tparams": cm.group("tparams") or "", "kind": cm.group("kind"),
                                           "path": path, "fields": fields, "code": text[cstart:end + 1].strip(), "file_text": text.strip()})
    methods = []
    for mm in TS_METHOD_RE.finditer(text):
        end = match_brace(text, mm.end() - 1)
        start, _ = leading_block(text, mm.start())
        methods.append({"name": mm.group("name"), "params": mm.group("params"), "ret": (mm.group("ret") or "").strip(),
                        "sig_start": mm.start(), "start": start, "end": end + 1})
    for hm in HTTP_CALL_RE.finditer(text):
        open_idx = hm.end() - 1
        args = split_top_level(text[open_idx + 1:match_brace(text, open_idx, "(", ")")])
        m = next((m for m in methods if m["sig_start"] <= hm.start() < m["end"]), None)
        if not args or not m: continue
        verb = hm.group("verb").upper()
        mtext = text[m["start"]:m["end"]]
        route, dyn = normalize_url_expr(args[0])
        body_expr = args[1] if verb in ("POST", "PUT", "PATCH") and len(args) > 1 else None
        body_type, sent_keys = resolve_body(mtext, m["params"], body_expr) if body_expr else (None, [])
        ret = re.fullmatch(r"Observable<(.+)>", m["ret"])
        calls.append({
            "service": service["name"] if service else "", "service_path": path, "method": m["name"],
            "verb": verb, "url_expr": args[0], "route": route.lower(), "dynamic_tail": dyn,
            "query_names": sorted(set(re.findall(r"[?&](\w+)=", mtext))),
            "body_expr": body_expr, "body_type": body_type, "sent_keys": sent_keys,
            "response_type": ret.group(1) if ret else (m["ret"] or "void"),
            "code": dedent(mtext), "ctor": service["ctor"] if service else "",
            "feature": (re.search(r"/app/([^/]+)/", "/" + path.split("src/", 1)[-1]) or [None, ""])[1]})
    return models, service, calls

# ---------- run on the whole repo ----------
texts, cs_classes, endpoints, ts_models, ts_services, calls = {}, [], [], [], [], []
for f in files:
    rel = f.relative_to(ROOT).as_posix()
    if f.suffix == ".cs":
        texts[rel] = read_text(f)
        c, e = parse_cs_file(texts[rel], rel); cs_classes += c; endpoints += e
    elif f.suffix == ".ts" and not rel.endswith(".spec.ts"):
        texts[rel] = read_text(f)
        m, s, c = parse_ts_file(texts[rel], rel); ts_models += m; calls += c
        if s: ts_services.append(s)

df_endpoints = pd.DataFrame(endpoints)
df_calls     = pd.DataFrame(calls)
print(f"{len(endpoints)} backend endpoints | {len(calls)} frontend HTTP calls | {len(ts_models)} TS models | {len(cs_classes)} C# classes")
display(df_endpoints[["controller", "action", "verb", "route", "body_type", "query_params", "response_type", "auth"]])
display(df_calls[["service", "method", "verb", "url_expr", "route", "body_type", "sent_keys", "response_type"]])

9 backend endpoints | 8 frontend HTTP calls | 12 TS models | 44 C# classes


,controller,action,verb,route,body_type,query_params,response_type,auth
0,AccountsController,Post,POST,/api/accounts/sign-up,Register,[],NaN,NaN
1,AccountsController,Post,POST,/api/accounts/sign-in,Login,[],NaN,NaN
2,ProductController,AddPost,POST,/api/product/addproduct,ProductVM,[],void,AllowAnonymous
3,UsersController,GetUsers,GET,/api/users,NaN,"[page, pageSize]",PaginationHelper<UserResponse>,"Authorize(Policy = ""ApiUser"")"
4,UsersController,GetUser,GET,/api/users/{id},NaN,[],NaN,"Authorize(Policy = ""ApiUser"")"
5,UsersController,GetUserForEdit,GET,/api/users/full/{id},NaN,[],NaN,"Authorize(Policy = ""ApiUser"")"
6,UsersController,UpdateUser,PUT,/api/users/{update},UpdateUserRequest,[],NaN,"Authorize(Policy = ""ApiUser"")"
7,UsersController,AddUser,POST,/api/users/{add},AddUserRequest,[],NaN,"Authorize(Policy = ""ApiUser"")"
8,UsersController,DeleteUser,DELETE,/api/users/{id},NaN,[],NaN,"Authorize(Policy = ""ApiUser"")"


,service,method,verb,url_expr,route,body_type,sent_keys,response_type
0,AccountService,SignUp,POST,'/api/accounts/sign-up',/api/accounts/sign-up,any,"[Email, Password, Data]",number
1,AccountService,SignIn,POST,'/api/accounts/sign-in',/api/accounts/sign-in,any,"[Email, Password]",Response
2,UserApiService,GetUserById,GET,'/api/users/' + id,/api/users/{},NaN,[],UserResponse
3,UserApiService,GetUserForEditById,GET,'/api/users/full/' + id,/api/users/full/{},NaN,[],User
4,UserApiService,GetUsers,GET,'/api/users' + parameters,/api/users,NaN,[],PaginationHelper<UserResponse>
5,UserApiService,DeleteUser,DELETE,'/api/users/' + id,/api/users/{},NaN,[],void
6,UserApiService,AddUser,POST,'/api/users/add',/api/users/add,AddUserRequest,[],void
7,UserApiService,UpdateUser,PUT,'/api/users/update',/api/users/update,UpdateUserRequest,[],void


## 5. Type mapping, DTO ⇄ TypeScript matching, endpoint ⇄ call matching

In [5]:
TS_SCALARS = {"string": "string", "char": "string", "guid": "string", "datetime": "string", "datetimeoffset": "string",
              "dateonly": "string", "timeonly": "string", "timespan": "string", "uri": "string",
              "bool": "boolean", "boolean": "boolean",
              "byte": "number", "sbyte": "number", "short": "number", "ushort": "number", "int": "number", "uint": "number",
              "long": "number", "ulong": "number", "float": "number", "double": "number", "decimal": "number",
              "object": "any", "dynamic": "any"}
LIST_TYPES = {"list", "ilist", "ienumerable", "icollection", "ireadonlylist", "ireadonlycollection", "hashset", "iset"}
DICT_TYPES = {"dictionary", "idictionary", "ireadonlydictionary"}

def is_nullable(t):
    t = re.sub(r"\s+", "", t); return t.endswith("?") or t.startswith("Nullable<")

def cs_to_ts(t):
    t = re.sub(r"\s+", "", t).rstrip("?")
    m = re.fullmatch(r"Nullable<(.+)>", t)
    if m: return cs_to_ts(m.group(1))
    if t == "byte[]": return "string"                                   # base64 in JSON
    if t.endswith("[]"): return cs_to_ts(t[:-2]) + "[]"
    m = re.fullmatch(r"([\w.]+)<(.+)>", t)
    if m:
        base, args = m.group(1).split(".")[-1], split_top_level(m.group(2))
        if base.lower() in LIST_TYPES: return cs_to_ts(args[0]) + "[]"
        if base.lower() in DICT_TYPES and len(args) == 2: return "{ [key: string]: " + cs_to_ts(args[1]) + " }"
        return f"{base}<{', '.join(cs_to_ts(a) for a in args)}>"
    name = t.split(".")[-1]
    return TS_SCALARS.get(name.lower(), name)

def gen_ts(cls, style="class", camel=False):
    """Rule-based C# -> TypeScript model. Defaults mimic this project's convention (export class, PascalCase)."""
    lines = []
    for p in cls["props"]:
        nm = p["name"][0].lower() + p["name"][1:] if camel else p["name"]
        ts = cs_to_ts(p["type"]) + (" | null" if is_nullable(p["type"]) else "")
        lines.append(f"    {nm}: {ts};")
    return f"export {style} {cls['name']}{cls['tparams']} {{\n" + "\n".join(lines) + "\n}\n"

norm_ts = lambda t: re.sub(r"\s+|\|null|\|undefined", "", t)

# --- which C# classes are DTOs? (referenced by an endpoint signature, or named/placed like a DTO)
DTO_HINT = re.compile(r"(DTO|Dto|VM\b|ViewModel|Request|Response|Helper|TransportModels|ModelsView)")
def type_tokens(t): return set(re.findall(r"[A-Za-z_]\w*", t or ""))
referenced = set().union(*[type_tokens(e["body_type"]) | type_tokens(e["response_type"]) for e in endpoints]) if endpoints else set()
cs_dtos = [c for c in cs_classes if not c["is_controller"] and c["props"]
           and (c["name"] in referenced or DTO_HINT.search(c["path"] + " " + c["name"]))]
cs_by_name = {c["name"]: c for c in cs_dtos}
ts_by_name = {m["name"]: m for m in ts_models}

def match_dtos(cs_list, ts_list):
    pairs, used_ts, used_cs = [], set(), set()
    by_name = {m["name"].lower(): m for m in ts_list}
    for c in cs_list:                                           # 1) same class name
        m = by_name.get(c["name"].lower())
        if m: pairs.append((c, m, "name")); used_ts.add(m["name"]); used_cs.add(c["name"])
    rest_t = [m for m in ts_list if m["name"] not in used_ts]
    for c in (c for c in cs_list if c["name"] not in used_cs):  # 2) same field set (e.g. UserEditResponse <-> User)
        want = {p["name"].lower() for p in c["props"]}
        cands = [m for m in rest_t if want and {f["name"].lower() for f in m["fields"]} == want]
        if len(cands) == 1: pairs.append((c, cands[0], "structure")); rest_t.remove(cands[0])
    return pairs

def model_diff(c, m):
    cp = {p["name"].lower(): p for p in c["props"]}
    tf = {f["name"].lower(): f for f in m["fields"]}
    types = []
    for k in cp.keys() & tf.keys():
        want, got = norm_ts(cs_to_ts(cp[k]["type"])), norm_ts(tf[k]["type"])
        if want != got and "any" not in (want, got): types.append((cp[k]["name"], cp[k]["type"], want, got))
    return {"missing_in_ts": [cp[k]["name"] for k in cp.keys() - tf.keys()],
            "extra_in_ts":   [tf[k]["name"] for k in tf.keys() - cp.keys()],
            "type_mismatch": types}

dto_pairs = match_dtos(cs_dtos, ts_models)
df_dto = pd.DataFrame([{"csharp": c["name"], "typescript": m["name"], "matched_by": how,
                        "cs_file": c["path"], "ts_file": m["path"], **{k: v for k, v in model_diff(c, m).items()}}
                       for c, m, how in dto_pairs])
print("DTO <-> TypeScript model pairs:"); display(df_dto)
print("C# DTOs without a TypeScript model:", sorted(set(cs_by_name) - {c["name"] for c, _, _ in dto_pairs}))

# --- endpoint <-> frontend call
def match_call(call, eps):
    """Best backend endpoint for a frontend call: same verb, same segments, backend {} may match any segment."""
    cr, best, best_key = call["route"].strip("/").split("/"), None, None
    for i, ep in enumerate(eps):
        if ep["verb"] != call["verb"]: continue
        br = ep["route_norm"].strip("/").split("/")
        if len(br) == len(cr) and all(b == c or b == "{}" for b, c in zip(br, cr)):
            key = (sum(b == c and b != "{}" for b, c in zip(br, cr)), -sum(b == "{}" and c != "{}" for b, c in zip(br, cr)))
            if best_key is None or key > best_key: best, best_key = i, key
    if best is None: return None, None
    return best, ("high" if eps[best]["route_norm"].strip("/") == call["route"].strip("/") else "medium")

matches = []
for ci, call in enumerate(calls):
    ei, conf = match_call(call, endpoints)
    if ei is not None: matches.append({"call_i": ci, "ep_i": ei, "confidence": conf})
df_matches = pd.DataFrame([{"controller": endpoints[m["ep_i"]]["controller"], "action": endpoints[m["ep_i"]]["action"],
                            "backend": f'{endpoints[m["ep_i"]]["verb"]} {endpoints[m["ep_i"]]["route"]}',
                            "service": calls[m["call_i"]]["service"], "method": calls[m["call_i"]]["method"],
                            "frontend": f'{calls[m["call_i"]]["verb"]} {calls[m["call_i"]]["route"]}',
                            "confidence": m["confidence"]} for m in matches])
print(f"{len(matches)} of {len(calls)} frontend calls matched a backend endpoint"); display(df_matches)

DTO <-> TypeScript model pairs:


,csharp,typescript,matched_by,cs_file,ts_file,missing_in_ts,extra_in_ts,type_mismatch
0,AddUserRequest,AddUserRequest,name,ACWA.Services/TransportModels/User/Request/AddUserRequest.cs,ACWAClientApp/src/app/users/shared/add-user-request.model.ts,[],[],[]
1,UpdateUserRequest,UpdateUserRequest,name,ACWA.Services/TransportModels/User/Request/UpdateUserRequest.cs,ACWAClientApp/src/app/users/shared/update-user-request.model.ts,[],[],[]
2,UserResponse,UserResponse,name,ACWA.Services/TransportModels/User/Response/UserResponse.cs,ACWAClientApp/src/app/users/shared/user-response.model.ts,[],[],[]
3,PaginationHelper,PaginationHelper,name,ACWA.Web/Extensions/PaginationHelper.cs,ACWAClientApp/src/app/users/shared/pagination-helper.model.ts,[],[],[]
4,UserEditResponse,User,structure,ACWA.Services/TransportModels/User/Response/UserEditResponse.cs,ACWAClientApp/src/app/users/shared/user.model.ts,[],[],[]


C# DTOs without a TypeScript model: ['CategoryDTO', 'CategoryVM', 'Login', 'ProductDTO', 'ProductVM', 'Register']
8 of 8 frontend calls matched a backend endpoint


,controller,action,backend,service,method,frontend,confidence
0,AccountsController,Post,POST /api/accounts/sign-up,AccountService,SignUp,POST /api/accounts/sign-up,high
1,AccountsController,Post,POST /api/accounts/sign-in,AccountService,SignIn,POST /api/accounts/sign-in,high
2,UsersController,GetUser,GET /api/users/{id},UserApiService,GetUserById,GET /api/users/{},high
3,UsersController,GetUserForEdit,GET /api/users/full/{id},UserApiService,GetUserForEditById,GET /api/users/full/{},high
4,UsersController,GetUsers,GET /api/users,UserApiService,GetUsers,GET /api/users,high
5,UsersController,DeleteUser,DELETE /api/users/{id},UserApiService,DeleteUser,DELETE /api/users/{},high
6,UsersController,AddUser,POST /api/users/{add},UserApiService,AddUser,POST /api/users/add,medium
7,UsersController,UpdateUser,PUT /api/users/{update},UserApiService,UpdateUser,PUT /api/users/update,medium


## 6. Consistency checks (also usable as a dataset validation step)

| Rule | Severity | What it finds |
|---|---|---|
| `frontend_call_without_endpoint` | error | service calls a route that no controller exposes |
| `http_verb_mismatch` | error | route exists, but under a different HTTP verb |
| `route_param_not_in_signature` | warning | `[HttpPut("{update}")]` declares a route parameter the action never uses (probably meant the literal `"update"`) |
| `body_field_mismatch` | warning | client sends a field the C# request class lacks (silently ignored by model binding) |
| `model_field_mismatch` | warning | C# DTO and TypeScript model differ in fields or types |
| `query_param_mismatch` | warning / info | query-string names the endpoint does not read (case-only differences are `info`: ASP.NET binds case-insensitively) |
| `response_type_mismatch` | warning | declared response types differ |
| `endpoint_without_frontend_call` | info | endpoint nobody calls (dead code or not built in the UI yet) |
| `dto_without_ts_model` | info | C# DTO used by the API with no TypeScript model |

In [6]:
def issue(sev, rule, detail, ep=None, call=None, fix=None):
    return {"severity": sev, "rule": rule, "detail": detail,
            "controller": ep["controller"] if ep else None, "action": ep["action"] if ep else None,
            "service": call["service"] if call else None, "method": call["method"] if call else None, "fix": fix}

def check_unmatched_call(call, eps):
    other = [e for e in eps if e["verb"] != call["verb"]
             and match_call({**call, "verb": e["verb"]}, [e])[0] is not None]
    if other:
        return [issue("error", "http_verb_mismatch", f'{call["service"]}.{call["method"]} uses {call["verb"]} {call["route"]}, '
                      f'but that route only accepts {", ".join(sorted({e["verb"] for e in other}))} '
                      f'({other[0]["controller"]}.{other[0]["action"]}).', other[0], call)]
    return [issue("error", "frontend_call_without_endpoint",
                  f'{call["service"]}.{call["method"]} calls {call["verb"]} {call["route"]} and no controller exposes it.', None, call)]

def check_pair(call, ep):
    out = []
    # query string names
    back_q = {q.lower(): q for q in ep["query_params"]}
    for q in call["query_names"]:
        if q.lower() not in back_q:
            out.append(issue("warning", "query_param_mismatch", f'Client sends ?{q}= but {ep["controller"]}.{ep["action"]} has no such query parameter.', ep, call))
        elif back_q[q.lower()] != q:
            out.append(issue("info", "query_param_mismatch", f'Client sends ?{q}= and the action declares {back_q[q.lower()]} (works: binding ignores case).', ep, call))
    # request body
    if ep["body_type"] and not call["body_expr"]:
        out.append(issue("error", "body_field_mismatch", f'{ep["action"]} expects a {ep["body_type"]} body but {call["method"]} sends none.', ep, call))
    elif call["body_expr"] and not ep["body_type"]:
        out.append(issue("warning", "body_field_mismatch", f'{call["method"]} sends a body but {ep["action"]} has no [FromBody] parameter.', ep, call))
    elif ep["body_type"] and ep["body_type"] in cs_by_name:
        sent = call["sent_keys"] or [f["name"] for f in ts_by_name.get(call["body_type"], {"fields": []})["fields"]]
        want = {p["name"].lower(): p["name"] for p in cs_by_name[ep["body_type"]]["props"]}
        if sent:
            extra = [k for k in sent if k.lower() not in want]
            if extra: out.append(issue("warning", "body_field_mismatch", f'{call["method"]} sends {extra} which {ep["body_type"]} does not have (ignored by the server).', ep, call))
            miss = [v for k, v in want.items() if k not in {s.lower() for s in sent}]
            if miss:  out.append(issue("info", "body_field_mismatch", f'{call["method"]} never sends {miss} (present in {ep["body_type"]}).', ep, call))
    # response type
    r_back, r_front = ep["response_type"], call["response_type"]
    if r_back not in (None, "void") and r_front not in ("void", "Response", "any") and re.sub(r"\s", "", r_back) != re.sub(r"\s", "", r_front):
        out.append(issue("warning", "response_type_mismatch", f'{ep["action"]} returns {r_back} but {call["method"]} declares {r_front}.', ep, call))
    return out

def check_endpoint(ep):
    names = {p["name"].lower() for p in ep["params"]}
    return [issue("warning", "route_param_not_in_signature",
                  f'Route "{ep["route"]}" declares {{{p}}} but {ep["action"]} has no parameter named "{p}"; '
                  f'{ep["verb"]} {ep["route"].replace("{" + p + "}", "anything")} would match too. '
                  f'If a literal segment was intended, write "{p}" without braces.', ep, None,
                  fix=("route_literal", p))
            for p in ep["route_params"] if p.lower() not in names]

def check_model_pair(c, m):
    d, out = model_diff(c, m), []
    if d["missing_in_ts"]: out.append(issue("warning", "model_field_mismatch", f'TypeScript {m["name"]} is missing {d["missing_in_ts"]} from C# {c["name"]}.'))
    if d["extra_in_ts"]:   out.append(issue("warning", "model_field_mismatch", f'TypeScript {m["name"]} has {d["extra_in_ts"]} that C# {c["name"]} lacks.'))
    for name, cs_t, want, got in d["type_mismatch"]:
        out.append(issue("warning", "model_field_mismatch", f'{c["name"]}.{name}: C# {cs_t} should be {want} in TypeScript, found {got}.'))
    return out

def check_all():
    out, matched_calls, matched_eps = [], {m["call_i"] for m in matches}, {m["ep_i"] for m in matches}
    for ci, call in enumerate(calls):
        if ci not in matched_calls: out += check_unmatched_call(call, endpoints)
    for m in matches: out += check_pair(calls[m["call_i"]], endpoints[m["ep_i"]])
    for ei, ep in enumerate(endpoints):
        out += check_endpoint(ep)
        if ei not in matched_eps:
            out.append(issue("info", "endpoint_without_frontend_call", f'{ep["verb"]} {ep["route"]} ({ep["controller"]}.{ep["action"]}) is not called by any Angular service.', ep))
    for c, m, _ in dto_pairs: out += check_model_pair(c, m)
    paired = {c["name"] for c, _, _ in dto_pairs}
    for name in sorted(referenced & set(cs_by_name) - paired):
        out.append(issue("info", "dto_without_ts_model", f'C# {name} is used by the API but has no TypeScript model (the client uses an untyped object or never calls it).'))
    return out

issues = check_all()
df_issues = pd.DataFrame(issues)
print(f"{len(df_issues)} findings"); display(df_issues.drop(columns="fix").sort_values(["severity", "rule"]))
df_issues.drop(columns="fix").to_csv(OUT_DIR / "consistency_report.csv", index=False)

7 findings


,severity,rule,detail,controller,action,service,method
4,info,dto_without_ts_model,C# Login is used by the API but has no TypeScript model (the clien...,NaN,NaN,NaN,NaN
5,info,dto_without_ts_model,C# ProductVM is used by the API but has no TypeScript model (the c...,NaN,NaN,NaN,NaN
6,info,dto_without_ts_model,C# Register is used by the API but has no TypeScript model (the cl...,NaN,NaN,NaN,NaN
1,info,endpoint_without_frontend_call,POST /api/product/addproduct (ProductController.AddPost) is not ca...,ProductController,AddPost,NaN,NaN
0,info,query_param_mismatch,Client sends ?pagesize= and the action declares pageSize (works: b...,UsersController,GetUsers,UserApiService,GetUsers
2,warning,route_param_not_in_signature,"Route ""/api/users/{update}"" declares {update} but UpdateUser has n...",UsersController,UpdateUser,NaN,NaN
3,warning,route_param_not_in_signature,"Route ""/api/users/{add}"" declares {add} but AddUser has no paramet...",UsersController,AddUser,NaN,NaN


## 7. Build the training examples

In [7]:
examples = []
def add_example(task_type, feature, prompt, response, source="real", **meta):
    examples.append({"task_type": task_type, "feature": feature, "source": source,
                     "prompt": prompt.strip(), "response": response.strip(), **meta})

def fence(lang, body): return f"```{lang}\n{dedent(body).rstrip()}\n```"
def hint(label, path): return f"\n{label}: `{path}`" if INCLUDE_PATH_HINT else ""
def feature_of_ts(path):
    m = re.search(r"/app/([^/]+)/", "/" + path.split("src/", 1)[-1]); return m.group(1) if m else "shared"

# ---- 7.1 / 7.2  file level: whole controller <-> whole Angular service (only clean 1:1 pairs) ----
ctrl_svcs, svc_ctrls = defaultdict(set), defaultdict(set)
for m in matches:
    ep, call = endpoints[m["ep_i"]], calls[m["call_i"]]
    ctrl_svcs[ep["controller_path"]].add(call["service_path"]); svc_ctrls[call["service_path"]].add(ep["controller_path"])
file_pairs = [(c, next(iter(s))) for c, s in ctrl_svcs.items() if len(s) == 1 and len(svc_ctrls[next(iter(s))]) == 1]
ctrl_feature = {e["controller_path"]: e["feature"] for e in endpoints}

for cpath, spath in file_pairs:
    feat = ctrl_feature[cpath]
    add_example("backend_to_frontend", feat,
        "Write the Angular service for this controller." + hint("Target file", spath) + "\n\n" + fence("csharp", texts[cpath]),
        fence("typescript", texts[spath]), controller=cpath, service=spath)
    add_example("frontend_to_backend", feat,
        "Write the API endpoint that this service expects." + hint("Target file", cpath) + "\n\n" + fence("typescript", texts[spath]),
        fence("csharp", texts[cpath]), controller=cpath, service=spath)
skipped = [c for c in ctrl_svcs if c not in {p[0] for p in file_pairs}]
print("file-level pairs:", [(Path(c).name, Path(s).name) for c, s in file_pairs], "| not 1:1:", skipped)

# ---- 7.3  DTO -> TypeScript model (response = the REAL model file; rule-based version is only a cross-check) ----
for c, m, how in dto_pairs:
    d = model_diff(c, m)
    agree = not (d["missing_in_ts"] or d["extra_in_ts"] or d["type_mismatch"])
    add_example("dto_to_ts", feature_of_ts(m["path"]),
        "Convert this C# class into the TypeScript model used by the Angular client. Follow the project's model conventions."
        + hint("Target file", m["path"])
        + (f"\nName the TypeScript class `{m['name']}`." if how == "structure" else "")      # name differs from the C# class
        + "\n\n" + fence("csharp", c["code"]),
        fence("typescript", m["file_text"]), csharp=c["path"], typescript=m["path"], matched_by=how,
        rule_based_agrees=agree)

# ---- 7.4  endpoint level: one action <-> one service method, both directions ----
def cs_context(ep):
    names = [n for n in type_tokens(ep["body_type"]) | type_tokens(ep["response_type"]) if n in cs_by_name]
    return "\n\n".join(dedent(cs_by_name[n]["code"]) for n in sorted(names))
def ts_context(call):
    names = [n for n in type_tokens(call["body_type"]) | type_tokens(call["response_type"]) if n in ts_by_name]
    return "\n\n".join(ts_by_name[n]["code"] for n in sorted(names))

for m in matches:
    ep, call = endpoints[m["ep_i"]], calls[m["call_i"]]
    ctx_cs, ctx_ts = cs_context(ep), ts_context(call)
    p1 = (f"Write the Angular service method for this ASP.NET Core endpoint.\n\nRoute: {ep['verb']} {ep['route']}"
          + (f"\nAuth: {ep['auth']}" if ep["auth"] else "") + f"\nService class: `{call['service']}` ({call['ctor']})\n\n"
          + "Controller:\n" + fence("csharp", ep["class_decl"] + "\n{ ... }") + "\n\nEndpoint:\n" + fence("csharp", dedent(ep["code"]))
          + ("\n\nRelated C# types:\n" + fence("csharp", ctx_cs) if ctx_cs else ""))
    add_example("endpoint_to_method", ep["feature"], p1, fence("typescript", call["code"]),
                endpoint=ep["id"], method=f'{call["service"]}.{call["method"]}', confidence=m["confidence"])
    p2 = (f"Write the ASP.NET Core controller action that this Angular service method calls.\n\n"
          f"Service method (`{call['service']}.{call['method']}`):\n" + fence("typescript", call["code"])
          + ("\n\nRelated TypeScript models:\n" + fence("typescript", ctx_ts) if ctx_ts else "")
          + f"\n\nController: `{ep['controller']}` with `[Route(\"{ep['base_route']}\")]`")
    add_example("method_to_endpoint", ep["feature"], p2, fence("csharp", dedent(ep["code"])),
                endpoint=ep["id"], method=f'{call["service"]}.{call["method"]}', confidence=m["confidence"])
print(len(examples), "examples so far")

file-level pairs: [('AccountsController.cs', 'account.service.ts'), ('UsersController.cs', 'user-api.service.ts')] | not 1:1: []
25 examples so far


### 7b. "Find the bug" examples

Two sources, both **verified by the checker from section 6** before they are kept:

1. **Real findings**: warnings the checker raised on your actual code (e.g. `{update}` instead of `"update"`). The fix is applied and the checker is re-run to confirm it disappears and the frontend call still matches.
2. **Mutations**: take a correct controller/service pair, break the client in a realistic way (misspelled route segment, wrong HTTP verb, dropped model field), and keep the example *only if the checker detects the break*.

In [8]:
BUG_PROMPT = ("The Angular service method below calls the ASP.NET Core endpoint below, but the two do not match. "
              "Find the problem and give the corrected code.")

# ---- real findings: route_param_not_in_signature, fix verified by re-parsing ----
for i in issues:
    if i["rule"] != "route_param_not_in_signature": continue
    ep = next(e for e in endpoints if e["controller"] == i["controller"] and e["action"] == i["action"])
    param = i["fix"][1]
    fixed_code = ep["code"].replace("{" + param + "}", param, 1)
    fixed_text = texts[ep["controller_path"]].replace(ep["code"], fixed_code, 1)
    _, new_eps = parse_cs_file(fixed_text, ep["controller_path"])
    new_ep = next((e for e in new_eps if e["action"] == ep["action"] and e["verb"] == ep["verb"]), None)
    partner = next((calls[m["call_i"]] for m in matches if endpoints[m["ep_i"]]["id"] == ep["id"]), None)
    still_ok = bool(new_ep) and not check_endpoint(new_ep) and (partner is None or match_call(partner, [new_ep]) == (0, "high"))
    if not still_ok: continue                                        # only keep fixes the checker confirms
    add_example("find_the_bug", ep["feature"],
        "This ASP.NET Core endpoint is called by the Angular client. Review its route and point out any problem.\n\n"
        + fence("csharp", dedent(ep["code"])),
        i["detail"] + "\n\nCorrected endpoint:\n" + fence("csharp", dedent(fixed_code)),
        source="real_finding", verified=True, rule=i["rule"], endpoint=ep["id"])

# ---- mutations ----
def mutate_url(code):
    new, n = re.subn(r"(/api/)([A-Za-z][\w-]*)", lambda m: m.group(1) + m.group(2)[:-1], code, count=1)
    return new if n else None

VERB_SWAP = {"get": "post", "post": "put", "put": "post", "delete": "get"}
def mutate_verb(code):
    m = re.search(r"this\.http\.(get|post|put|delete)\b", code)
    return (code[:m.start(1)] + VERB_SWAP[m.group(1)] + code[m.end(1):]) if m else None

def first_call(code_str):
    _, _, cs = parse_ts_file("export class X {\n" + code_str + "\n}\n", "mutation.ts")
    return cs[0] if cs else None

def detected(mut_call, ep_index):
    j, _ = match_call(mut_call, endpoints)
    return j is None or j != ep_index

n_mut = 0
for m in matches:
    ep, call = endpoints[m["ep_i"]], calls[m["call_i"]]
    for kind, mutate in (("url_typo", mutate_url), ("wrong_verb", mutate_verb)):
        if kind not in MUTATION_KINDS: continue
        bad_code = mutate(call["code"])
        bad = first_call(bad_code) if bad_code else None
        if not bad or not detected(bad, m["ep_i"]): continue          # keep only what the checker catches
        if kind == "url_typo":
            why = (f"The service calls `{bad['url_expr']}`, but {ep['controller']}.{ep['action']} is `{ep['verb']} {ep['route']}`. "
                   f"The original call used `{call['url_expr']}`; the controller segment is misspelled.")
        else:
            why = (f"{ep['controller']}.{ep['action']} accepts `{ep['verb']}` but the method calls `this.http.{bad['verb'].lower()}`. "
                   f"Use `this.http.{call['verb'].lower()}`.")
        add_example("find_the_bug", ep["feature"],
            BUG_PROMPT + "\n\nEndpoint:\n" + fence("csharp", dedent(ep["code"])) + "\n\nService method:\n" + fence("typescript", bad_code),
            why + "\n\nCorrected method:\n" + fence("typescript", call["code"]),
            source="mutation", verified=True, rule=kind, endpoint=ep["id"], method=f'{call["service"]}.{call["method"]}')
        n_mut += 1

for c, m, how in dto_pairs:                                              # drop_field: TS model loses a field the C# class has
    if "drop_field" not in MUTATION_KINDS or len(c["props"]) < 3 or not c["props"]: continue
    victim = [p["name"] for p in c["props"] if p["name"].lower() != "id"][-1]
    bad_text = re.sub(rf"(?m)^[ \t]*{victim}\??\s*:[^\n]*\n", "", m["file_text"] + "\n", count=1).strip()
    bad_models, _, _ = parse_ts_file(bad_text, m["path"])
    if not bad_models or victim not in model_diff(c, bad_models[0])["missing_in_ts"]: continue
    ts_type = cs_to_ts(next(p["type"] for p in c["props"] if p["name"] == victim))
    add_example("find_the_bug", feature_of_ts(m["path"]),
        "This TypeScript model is supposed to mirror the C# class that the API serialises. Find the mismatch and give the corrected model.\n\n"
        "C# class:\n" + fence("csharp", c["code"]) + "\n\nTypeScript model:\n" + fence("typescript", bad_text),
        f"`{victim}` exists on C# `{c['name']}` but is missing from the TypeScript `{m['name']}`, so the client never sees it. "
        f"Add `{victim}: {ts_type};`.\n\nCorrected model:\n" + fence("typescript", m["file_text"]),
        source="mutation", verified=True, rule="drop_field", csharp=c["path"], typescript=m["path"])
    n_mut += 1
print(f"{sum(e['task_type'] == 'find_the_bug' for e in examples)} find_the_bug examples "
      f"({sum(e['source'] == 'real_finding' for e in examples)} real, {n_mut} mutation-verified)")

23 find_the_bug examples (2 real, 21 mutation-verified)


### 7c. Feature scaffolding

A short spec is paired with every file of the feature: controller, C# DTOs, Angular service, TypeScript models. The spec is built from the parsed endpoints (rule-based); an LLM-written spec is a good upgrade later.

In [9]:
def endpoint_line(ep):
    bits = []
    if ep["query_params"]: bits.append("query: " + ", ".join(ep["query_params"]))
    if ep["body_type"]:    bits.append("body: " + ep["body_type"])
    if ep["auth"]:         bits.append("auth: " + ep["auth"])
    return f"- {ep['verb']} {ep['route']} -> {ep['response_type'] or 'IActionResult'}" + (f" ({'; '.join(bits)})" if bits else "")

lang_of = {".cs": "csharp", ".ts": "typescript"}
for feat in sorted({e["feature"] for e in endpoints}):
    eps = [e for e in endpoints if e["feature"] == feat]
    ctrl_paths = sorted({e["controller_path"] for e in eps})
    svc_paths  = sorted({calls[m["call_i"]]["service_path"] for m in matches if endpoints[m["ep_i"]]["feature"] == feat})
    if not svc_paths: continue                                   # backend-only feature: nothing to scaffold on the client
    cs_files = sorted({c["path"] for c, m, _ in dto_pairs if feature_of_ts(m["path"]) == feat}
                      | {cs_by_name[n]["path"] for e in eps for n in type_tokens(e["body_type"]) | type_tokens(e["response_type"]) if n in cs_by_name})
    ts_files = sorted({m["path"] for m in ts_models if feature_of_ts(m["path"]) == feat})
    ordered = ctrl_paths + cs_files + svc_paths + ts_files
    spec = (f'Add the "{feat}" feature to the project.\n\nAPI:\n' + "\n".join(endpoint_line(e) for e in eps)
            + "\n\nProduce the controller, the C# DTOs, the Angular service and the TypeScript models.")
    resp = "\n\n".join(f"### FILE: {p}\n" + fence(lang_of[Path(p).suffix], texts[p]) for p in ordered)
    add_example("feature_scaffold", feat, spec, resp, files=ordered)
print("scaffolded features:", [e["feature"] for e in examples if e["task_type"] == "feature_scaffold"])

scaffolded features: ['accounts', 'users']


## 8. Split by feature, check token budget, write JSONL

In [10]:
df_ex = pd.DataFrame(examples)
df_ex["id"]       = [hashlib.sha1((r.task_type + r.prompt).encode()).hexdigest()[:10] for r in df_ex.itertuples()]
df_ex["n_tokens"] = [approx_tokens(SYSTEM_MSG + p + r) for p, r in zip(df_ex.prompt, df_ex.response)]
df_ex["too_long"] = df_ex.n_tokens > MAX_TOKENS
df_ex["split"]    = df_ex.feature.map(lambda f: "test" if f in TEST_FEATURES else "train")   # whole features, never rows
df_ex = df_ex.drop_duplicates("id").reset_index(drop=True)

def to_messages(r):
    return {"messages": [{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": r.prompt},
                         {"role": "assistant", "content": r.response}],
            "meta": {"id": r.id, "task_type": r.task_type, "feature": r.feature, "source": r.source, "split": r.split}}

def write_jsonl(df, name):
    with open(OUT_DIR / name, "w", encoding="utf-8") as fh:
        for r in df.itertuples(): fh.write(json.dumps(to_messages(r), ensure_ascii=False) + "\n")
    print(f"{name:34s} {len(df):3d} examples")

write_jsonl(df_ex, "pairs_all.jsonl")
write_jsonl(df_ex[(df_ex.split == "train") & ~df_ex.too_long], "train.jsonl")
write_jsonl(df_ex[(df_ex.split == "test")  & ~df_ex.too_long], "test.jsonl")
for t, g in df_ex.groupby("task_type"): write_jsonl(g, f"pairs_{t}.jsonl")

print()
display(df_ex.pivot_table(index="task_type", columns="split", values="id", aggfunc="count", fill_value=0))
print("too long for MAX_TOKENS =", MAX_TOKENS, "->", df_ex[df_ex.too_long][["task_type", "feature", "n_tokens"]].to_dict("records"))
print("feature leakage check (must be empty):", set(df_ex[df_ex.split == "train"].feature) & set(df_ex[df_ex.split == "test"].feature))

pairs_all.jsonl                     50 examples
train.jsonl                         38 examples
test.jsonl                          11 examples
pairs_backend_to_frontend.jsonl      2 examples
pairs_dto_to_ts.jsonl                5 examples
pairs_endpoint_to_method.jsonl       8 examples
pairs_feature_scaffold.jsonl         2 examples
pairs_find_the_bug.jsonl            23 examples
pairs_frontend_to_backend.jsonl      2 examples
pairs_method_to_endpoint.jsonl       8 examples



split,test,train
task_type,,
backend_to_frontend,1,1
dto_to_ts,0,5
endpoint_to_method,2,6
feature_scaffold,1,1
find_the_bug,4,19
frontend_to_backend,1,1
method_to_endpoint,2,6


too long for MAX_TOKENS = 2048 -> [{'task_type': 'feature_scaffold', 'feature': 'users', 'n_tokens': 4923}]
feature leakage check (must be empty): set()


## 9. Review a few examples by hand (always do this before training)

In [11]:
def show(task_type, n=1):
    for r in df_ex[df_ex.task_type == task_type].head(n).itertuples():
        print(f"{'=' * 20} {r.task_type} | feature={r.feature} | source={r.source} | ~{r.n_tokens} tokens")
        print("--- PROMPT ---");   print(r.prompt[:1500] + (" ..." if len(r.prompt) > 1500 else ""))
        print("--- RESPONSE ---"); print(r.response[:1500] + (" ..." if len(r.response) > 1500 else ""))

show("find_the_bug", 2)

==================== find_the_bug | feature=users | source=real_finding | ~370 tokens
--- PROMPT ---
This ASP.NET Core endpoint is called by the Angular client. Review its route and point out any problem.

```csharp
// PUT: api/users/update
[HttpPut("{update}")]
public async Task<IActionResult> UpdateUser([FromBody] UpdateUserRequest model)
{
    if (!ModelState.IsValid)
    {
        return BadRequest(ModelState);
    }

    try
    {
        await _userService.UpdateUserAsync(model);
    }
    catch (Exception exp)
    {
        return BadRequest(exp);
    }

    return NoContent();
}
```
--- RESPONSE ---
Route "/api/users/{update}" declares {update} but UpdateUser has no parameter named "update"; PUT /api/users/anything would match too. If a literal segment was intended, write "update" without braces.

Corrected endpoint:
```csharp
// PUT: api/users/update
[HttpPut("update")]
public async Task<IActionResult> UpdateUser([FromBody] UpdateUserRequest model)
{
    if (!ModelState.IsVali

In [12]:
#RAG

In [13]:
chunks = []
def add_chunk(id_, kind, feature, path, summary, body, lang):
    text = f"{summary}\nFile: {path}\n\n{body}"        # summary + path inside the embedded text helps English queries
    chunks.append({"id": id_, "text": text, "kind": kind, "feature": feature, "path": path, "lang": lang,
                   "hash": hashlib.sha1(text.encode()).hexdigest()[:16]})

for ep in endpoints:
    add_chunk(f"ep::{ep['id']}", "endpoint", ep["feature"], ep["controller_path"],
              f"ASP.NET endpoint {ep['verb']} {ep['route']} ({ep['controller']}.{ep['action']}), auth: {ep['auth']}, "
              f"body: {ep['body_type']}, query: {ep['query_params']}", dedent(ep["code"]), "csharp")

for c in calls:
    add_chunk(f"call::{c['service']}.{c['method']}", "service_method", c["feature"], c["service_path"],
              f"Angular method {c['service']}.{c['method']} calls {c['verb']} {c['route']}", c["code"], "typescript")

for m in matches:                                       # backend + frontend together: the high-value chunk
    ep, c = endpoints[m["ep_i"]], calls[m["call_i"]]
    add_chunk(f"pair::{ep['id']}", "api_pair", ep["feature"], ep["controller_path"],
              f"{ep['verb']} {ep['route']} is implemented by {ep['controller']}.{ep['action']} and called by "
              f"{c['service']}.{c['method']}",
              "// BACKEND\n" + dedent(ep["code"]) + "\n\n// FRONTEND\n" + c["code"], "mixed")

for d in cs_dtos:
    add_chunk(f"cs::{d['name']}", "csharp_dto", "shared", d["path"], f"C# class {d['name']}", dedent(d["code"]), "csharp")
for t in ts_models:
    add_chunk(f"ts::{t['name']}", "ts_model", feature_of_ts(t["path"]), t["path"], f"TypeScript model {t['name']}", t["code"], "typescript")

for i in issues:                                        # known problems become searchable too
    if i["severity"] != "info":
        add_chunk(f"issue::{i['rule']}::{i['action'] or i['method']}", "finding", "shared", "consistency_report",
                  f"Known issue ({i['rule']})", i["detail"], "text")

print(len(chunks), "chunks")

50 chunks


In [14]:
# import chromadb, ollama
# db  = chromadb.PersistentClient(path="rag_db")
# col = db.get_or_create_collection("acwa", metadata={"hnsw:space": "cosine"})

# def embed(texts, prefix):                               # nomic-embed-text expects these prefixes
#     return ollama.embed(model="nomic-embed-text", input=[prefix + t for t in texts])["embeddings"]

# existing = col.get(include=["metadatas"])
# old = {i: m["hash"] for i, m in zip(existing["ids"], existing["metadatas"])}
# todo = [c for c in chunks if old.get(c["id"]) != c["hash"]]          # only new or changed chunks
# stale = set(old) - {c["id"] for c in chunks}                         # deleted code must disappear too

# for i in range(0, len(todo), 32):
#     b = todo[i:i + 32]
#     col.upsert(ids=[c["id"] for c in b], documents=[c["text"] for c in b],
#                embeddings=embed([c["text"] for c in b], "search_document: "),
#                metadatas=[{k: c[k] for k in ("kind", "feature", "path", "lang", "hash")} for c in b])
# if stale: col.delete(ids=list(stale))
# print(f"embedded {len(todo)}, deleted {len(stale)}, total {col.count()}")

In [15]:
### Fine-tune with LoRA (run on a GPU machine or Colab)

In [16]:
# !pip install unsloth trl datasets
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"   # use ...-3B-... if you have < 12 GB VRAM
MAX_SEQ    = 2048                                           # must be >= MAX_TOKENS (2048) + headroom

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

ds = load_dataset("json", data_files={"train": "out/train.jsonl", "test": "out/test.jsonl"})
ds = ds.map(lambda r: {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)},
            remove_columns=ds["train"].column_names)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["test"],        # test = held-out feature(s)
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=3, learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
        optim="adamw_8bit", weight_decay=0.01, logging_steps=1,
        eval_strategy="epoch", save_strategy="no", seed=42, output_dir="out/lora_run", report_to="none"))

# compute loss only on the assistant answer, not on the prompt (Qwen / ChatML markers)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

trainer.train()
model.save_pretrained("out/lora"); tokenizer.save_pretrained("out/lora")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/chipmunk/Desktop/project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1103 18:29:01.060000 127128 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1103 18:29:01.080000 127128 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.68 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:00<00:00, 602.85it/s]


Unsloth: Offloading embeddings to RAM to save 1.02 GB.


Unsloth 2026.10.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


... (6 outputs left out)

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.565499,0.600963
2,0.545488,0.542030
3,0.812042,0.531225


Filter: 100%|██████████| 11/11 [00:00<00:00, 4404.10 examples/s]
Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in out/lora/tokenizer_config.json.


('out/lora/tokenizer_config.json',
 'out/lora/chat_template.jinja',
 'out/lora/tokenizer.json')

In [17]:
model.save_pretrained_gguf("out/gguf", tokenizer, quantization_method="q4_k_m")   # merges LoRA + quantizes

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in out/gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /home/chipmunk/.cache/huggingface/hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 10.12it/s]


Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `out/gguf`: 100%|██████████| 4/4 [00:14<00:00,  3.67s/it]


Successfully copied all 4 files from cache to `out/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:14<00:00,  3.58s/it]


Unsloth: Merge process complete. Saved to `/home/chipmunk/Desktop/project/out/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['out/gguf_gguf/Qwen2.5-Coder-7B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['out/gguf_gguf/Qwen2.5-Coder-7B-Instruct.Q4_K_M.gguf']
Unsloth: example us

{'save_directory': 'out/gguf',
 'gguf_directory': 'out/gguf_gguf',
 'gguf_files': ['out/gguf_gguf/Qwen2.5-Coder-7B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'out/gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
from pathlib import Path
from llama_cpp import Llama

GGUF = next(Path("out").rglob("*.gguf"))      # picks the file found above
print("using:", GGUF)

llm = Llama(model_path=str(GGUF), n_ctx=8192, n_gpu_layers=-1, verbose=False)

SYSTEM_MSG = ("You are a coding assistant for the ACWA project. Stack: ASP.NET Core Web API (C#) and an "
              "Angular client that uses the legacy @angular/http service style. Follow the project's conventions.")

def ask(prompt, max_tokens=1500):
    r = llm.create_chat_completion(
        messages=[{"role": "system", "content": SYSTEM_MSG},
                  {"role": "user", "content": prompt}],
        temperature=0.2, max_tokens=max_tokens)
    return r["choices"][0]["message"]["content"]

print(ask("Convert this C# class into the TypeScript model used by the Angular client. "
          "Follow the project's model conventions.\n\n"
          "```csharp\npublic class OrderDTO { public Guid Id { get; set; } public decimal Total { get; set; } "
          "public List<string> Items { get; set; } }\n```"))

In [ ]:
print(ask('''Add the "orders" feature to the project.

API:
- GET /api/orders -> List<OrderDTO>
- GET /api/orders/{id} -> OrderDTO
- POST /api/orders -> OrderDTO (body: OrderDTO; auth: Authorize)

Produce the controller, the C# DTOs, the Angular service and the TypeScript models.''', max_tokens=2500))